## E.coli
P Wang, et al., Robust growth of Escherichia coli. Curr. Biol. 20, 1099–1103 (2010).

In [1]:
# include packages
using LinearAlgebra
using DifferentialEquations, Plots
using BenchmarkTools
using HypothesisTests
default(grid=false,legend=false, size=(400,320), dpi=300, tickfontsize=9, lw=2, lc=:gray, labelfontsize=9, tickdir=:out, fontfamily="Arial", legendfontsize=7, markercolor=:indianred, thickness_scaling=2)

mydir = "../helper/"
include(mydir * "base.jl")
include(mydir * "optimization.jl")
include(mydir * "misc.jl")
include(mydir * "myplot.jl")

plot_output (generic function with 3 methods)

In [4]:
using DelimitedFiles
# INSERT YOUR OWN DATA DIRECTORY AND FILENAME HERE
# E. coli MG1655 (CGSC 6300)/20090131/xy01c1/ch1_cell0.dat
data_dir = "../data/complete data Curr Biol 20, 1099-1103 (2010)/E. coli MG1655 (CGSC 6300)/"
DT = 1/60; # convert to hours
DL = 0.0645; # convert to um

In [5]:
# INSERT SELECTED FILES AND TIME WINDOWS HERE.

select_filename =[
    "20090131/xy01c1/ch0_cell0.dat",
    "20090131/xy01c1/ch1_cell0.dat",
    "20090131/xy01c1/ch2_cell0.dat",
    "20090131/xy01c1/ch3_cell0.dat",
    "20090131/xy01c1/ch4_cell0.dat",
    "20090131/xy01c1/ch5_cell0.dat",
    "20090131/xy01c1/ch6_cell0.dat",
    "20090131/xy02c1/ch0_cell0.dat",
    "20090131/xy03c1/ch0_cell0.dat",
    "20090131/xy04c1/ch0_cell0.dat",
    "20090131/xy04c1/ch1_cell0.dat",
    "20090131/xy05c1/ch0_cell0.dat",
    "20090131/xy05c1/ch1_cell0.dat",
    "20090131/xy05c1/ch2_cell0.dat",
    "20090131/xy05c1/ch3_cell0.dat",
    "20090131/xy06c1/ch0_cell0.dat",
    "20090131/xy06c1/ch1_cell0.dat",
    "20090131/xy06c1/ch2_cell0.dat",
    "20090131/xy06c1/ch3_cell0.dat",
    "20090131/xy07c1/ch0_cell0.dat",
    "20090131/xy07c1/ch1_cell0.dat",
    "20090131/xy07c1/ch2_cell0.dat",
    "20090131/xy07c1/ch3_cell0.dat",
    "20090131/xy07c1/ch4_cell0.dat",
    "20090131/xy07c1/ch5_cell0.dat",
    "20090131/xy08c1/ch0_cell0.dat",
    "20090131/xy08c1/ch1_cell0.dat",
    "20090131/xy08c1/ch2_cell0.dat",
    "20090131/xy08c1/ch3_cell0.dat",
    "20090131/xy08c1/ch4_cell0.dat",
    "20090131/xy09c1/ch0_cell0.dat",
    "20090131/xy09c1/ch1_cell0.dat",
    "20090131/xy09c1/ch2_cell0.dat",
    "20090131/xy09c1/ch3_cell0.dat",
    "20090131/xy10c1/ch0_cell0.dat",
    "20090131/xy10c1/ch1_cell0.dat",
    "20090131/xy10c1/ch2_cell0.dat",
    "20090131/xy11c1/ch0_cell0.dat",
    "20090131/xy12c1/ch0_cell0.dat",
    "20090131/xy12c1/ch1_cell0.dat",
    "20090131/xy12c1/ch2_cell0.dat",
    "20090131/xy12c1/ch3_cell0.dat",
    "20090131/xy12c1/ch4_cell0.dat",
    "20090131/xy12c1/ch5_cell0.dat",
    "20090131/xy12c1/ch6_cell0.dat",
    "20090131/xy12c1/ch7_cell0.dat",
    "20090131/xy13c1/ch0_cell0.dat",
    "20090131/xy13c1/ch1_cell0.dat",
    "20090131/xy13c1/ch2_cell0.dat",
    "20090131/xy14c1/ch0_cell0.dat",
]

select_time_all = [
    [(0.0, 11.0), (12.6, 26.8), (33.3, 46.8), (51.6, 60.8)],
    [(0.0, 20.8), (21.6, 34.3)],
    [(0.0, 24.0),(27.0,46.0)],
    [(52.,65.0)],
    [(58.,65.0)],
    [(51.5,64.0)],
    [(53.0,64.3)],
    [(0.0,18.55), (19.5,31.0)],
    [(0.0,29.5)],
    [(0.0,16.0), (21.0, 43.0)],
    [(0.0,4.0), (5.0, 21.8), (26.5,46.0)],
    [(0.0,8.5), (9.5, 32.8)],
    [(0.0,25.5), (26.2, 32.5)],
    [(0.0,20.1)],
    [(55.0,65.0)],
    [(0.0,13.5),(18.5,34.0)],
    [(0.0,27.9)],
    [(0.0,19.0),(22.5,34.5)],
    [(48.0,65.0)],
    [(0.0,14.2),(15.5,30.5)],
    [(0.0,12.5),(13.3,19.2),(22.0,48.0),(50.0,60.5)],
    [(1.3,22.0),(25.5,49.8),(52.5,65.0)],
    [(37.0,54.0)],
    [(40.2,61.0)],
    [(50.0,64.5)],
    [(0.0,40.5),(41.8,55.2)],
    [(0.0,39.5)],
    [(0.0, 21.0),(24.2,40.0)],
    [(48.0,65.0)],
    [(62.0,65.0)],
    [(4.5, 34.5), (36.5,56.0)],
    [(0.0,40.0)],
    [(0.0,4.3),(5.3,13.0),(13.5,44.0),(46.0,61.6)],
    [(58.0,65.0)],
    [(2.2,22.0)],
    [(0.0,18.5),(37.0,54.0)],
    [(58.0,65.0)],
    [(1.3,38.0)],
    [(0.,24.0),(31.0,39.4),(42.3,60.0)],
    [(41.5,65.0)],
    [(43.0,64.5)],
    [(43.6,65.0)],
    [(50.0,65.0)],
    [(44.0,65.0)],
    [(50.0,65.0)],
    [(50.0,65.0)],
    [(0.1,7.0),(8.2,43.4)],
    [(39.0,44.8),(50.4,63.6)],
    [(53.5,60.0)],
    [(0.0,50.0)]
]

# annotate
Lobs_ = []
tobs_ = []
for idx in eachindex(select_filename)
    tab = readdlm(data_dir * select_filename[idx]);
    varnames = convert(Vector{String}, tab[1,:]);
    tobs = tab[2:end,findfirst(varnames .== "index")] * DT;
    Lobs = tab[2:end,findfirst(varnames .== "length")] * DL;
    Lobs_idx, tobs_idx = filter_exp_traj(Lobs, tobs; select_time = select_time_all[idx] );
    Lobs_ = vcat(Lobs_, Lobs_idx)
    tobs_ = vcat(tobs_, tobs_idx)
end
Lobs_0131 = convert.(Vector{Float64}, Lobs_);
tobs_0131 = convert.(Vector{Float64}, tobs_);


In [6]:
# INSERT SELECTED FILES AND TIME WINDOWS HERE.

select_filename =[
    "20090129/xy01/ch0_cell0.dat",
    "20090129/xy01/ch1_cell0.dat",
    "20090129/xy01/ch2_cell0.dat",
    "20090129/xy01/ch3_cell0.dat",
    "20090129/xy01/ch4_cell0.dat",
    "20090129/xy01/ch5_cell0.dat",
    "20090129/xy01/ch6_cell0.dat",
    "20090129/xy01/ch7_cell0.dat",
    "20090129/xy01/ch8_cell0.dat",
    "20090129/xy01/ch9_cell0.dat",
    "20090129/xy01/ch10_cell0.dat",
    "20090129/xy02/ch0_cell0.dat",
    "20090129/xy02/ch1_cell0.dat",
    "20090129/xy02/ch2_cell0.dat",
    "20090129/xy02/ch3_cell0.dat",
    "20090129/xy02/ch4_cell0.dat",
    "20090129/xy02/ch5_cell0.dat",
    "20090129/xy02/ch6_cell0.dat",
    "20090129/xy02/ch7_cell0.dat",
    "20090129/xy02/ch8_cell0.dat",
    "20090129/xy03/ch0_cell0.dat",
    "20090129/xy03/ch1_cell0.dat",
    "20090129/xy03/ch2_cell0.dat",
    "20090129/xy03/ch3_cell0.dat",
    "20090129/xy03/ch4_cell0.dat",
    "20090129/xy03/ch5_cell0.dat",
    "20090129/xy03/ch6_cell0.dat",
    "20090129/xy03/ch7_cell0.dat",
    "20090129/xy03/ch8_cell0.dat",
    "20090129/xy03/ch9_cell0.dat",
    "20090129/xy03/ch10_cell0.dat",
    "20090129/xy04/ch0_cell0.dat",
    "20090129/xy04/ch1_cell0.dat",
    "20090129/xy04/ch2_cell0.dat",
    "20090129/xy04/ch3_cell0.dat",
    "20090129/xy04/ch4_cell0.dat",
    "20090129/xy04/ch5_cell0.dat",
    "20090129/xy04/ch6_cell0.dat",
    "20090129/xy04/ch7_cell0.dat",
    "20090129/xy04/ch8_cell0.dat",
    "20090129/xy04/ch9_cell0.dat",
    "20090129/xy05/ch0_cell0.dat",
    "20090129/xy05/ch1_cell0.dat",
    "20090129/xy05/ch2_cell0.dat",
    "20090129/xy05/ch3_cell0.dat",
    "20090129/xy05/ch4_cell0.dat",
    "20090129/xy05/ch5_cell0.dat",
    "20090129/xy05/ch6_cell0.dat",
    "20090129/xy05/ch7_cell0.dat",
    "20090129/xy05/ch8_cell0.dat",
    "20090129/xy05/ch9_cell0.dat",
    "20090129/xy05/ch10_cell0.dat",
    "20090129/xy05/ch11_cell0.dat",
    "20090129/xy05/ch12_cell0.dat",
    "20090129/xy06/ch0_cell0.dat",
    "20090129/xy06/ch1_cell0.dat",
    "20090129/xy06/ch2_cell0.dat",
    "20090129/xy06/ch3_cell0.dat",
    "20090129/xy06/ch4_cell0.dat",
    "20090129/xy06/ch5_cell0.dat",
    "20090129/xy06/ch6_cell0.dat",
    "20090129/xy06/ch7_cell0.dat",
    "20090129/xy06/ch8_cell0.dat",
    "20090129/xy06/ch9_cell0.dat",
    "20090129/xy07/ch0_cell0.dat",
    "20090129/xy07/ch1_cell0.dat",
    "20090129/xy07/ch2_cell0.dat",
    "20090129/xy07/ch3_cell0.dat",
    "20090129/xy07/ch4_cell0.dat",
    "20090129/xy07/ch5_cell0.dat",
    "20090129/xy07/ch6_cell0.dat",
    "20090129/xy07/ch7_cell0.dat",
    "20090129/xy07/ch8_cell0.dat",
    "20090129/xy07/ch10_cell0.dat",
    "20090129/xy07/ch11_cell0.dat",
    "20090129/xy08/ch0_cell0.dat",
    "20090129/xy08/ch1_cell0.dat",
    "20090129/xy08/ch3_cell0.dat",
    "20090129/xy08/ch4_cell0.dat",
    "20090129/xy08/ch5_cell0.dat",
    "20090129/xy08/ch6_cell0.dat",
    "20090129/xy08/ch7_cell0.dat",
    "20090129/xy08/ch8_cell0.dat",
    "20090129/xy08/ch9_cell0.dat",
    "20090129/xy08/ch10_cell0.dat",
    "20090129/xy09/ch0_cell0.dat",
    "20090129/xy09/ch1_cell0.dat",
    "20090129/xy09/ch2_cell0.dat",
    "20090129/xy09/ch3_cell0.dat",
    "20090129/xy09/ch4_cell0.dat",
    "20090129/xy09/ch5_cell0.dat",
    "20090129/xy09/ch6_cell0.dat",
    "20090129/xy09/ch7_cell0.dat",
    "20090129/xy09/ch8_cell0.dat",
    "20090129/xy09/ch9_cell0.dat",
    "20090129/xy09/ch10_cell0.dat",
    "20090129/xy09/ch11_cell0.dat",
    "20090129/xy10/ch0_cell0.dat",
    "20090129/xy10/ch1_cell0.dat",
    "20090129/xy10/ch2_cell0.dat",
    "20090129/xy10/ch3_cell0.dat",
    "20090129/xy10/ch4_cell0.dat",
    "20090129/xy10/ch5_cell0.dat",
    "20090129/xy10/ch6_cell0.dat",
    "20090129/xy10/ch7_cell0.dat",
    "20090129/xy10/ch8_cell0.dat",
    "20090129/xy10/ch9_cell0.dat",
    "20090129/xy10/ch10_cell0.dat",
    "20090129/xy10/ch11_cell0.dat",
    "20090129/xy10/ch12_cell0.dat",
    "20090129/xy10/ch13_cell0.dat",
    "20090129/xy11/ch0_cell0.dat",
    "20090129/xy11/ch1_cell0.dat",
    "20090129/xy11/ch2_cell0.dat",
    "20090129/xy11/ch3_cell0.dat",
    "20090129/xy11/ch4_cell0.dat",
    "20090129/xy11/ch5_cell0.dat",
    "20090129/xy11/ch6_cell0.dat",
    "20090129/xy11/ch7_cell0.dat",
    "20090129/xy11/ch8_cell0.dat",
    "20090129/xy11/ch9_cell0.dat",
    "20090129/xy11/ch10_cell0.dat",
    "20090129/xy12/ch0_cell0.dat",
    "20090129/xy12/ch1_cell0.dat",
    "20090129/xy12/ch2_cell0.dat",
    "20090129/xy12/ch3_cell0.dat",
    "20090129/xy12/ch4_cell0.dat"]

    select_time_all = [
    [(0.0, 11.41), (12.31, 15.97)],
    [(0.0, 19.77)], 
    [(0.0, 15.35)], 
    [(0.0, 2.57), (3.18, 13.49), (14.08, 15.88), (16.32, 19.07)],
    [(0.0, 18.76)], 
    [(0.0, 5.16), (5.68, 16.35), (16.79, 22.47), (22.78, 27.31)], 
    [(0.0, 23.55)],
    [(0.0, 27.31)], 
    [(0.0, 17.96), (18.64, 24.04)], 
    [(0.0, 22.71)],
    [(25.78, 27.31)],
    [(0.0, 20.20)],
    [(0.0, 27.31)],
    [(0.0, 26.56)],
    [(0.0, 23.30), (23.98, 27.31)],
    [(0.0, 27.31)],
    [(0.0, 18.52), (18.88, 20.39)],
    [(0.0, 27.31)],
    [(0.0, 26.67)],
    [(10.66, 27.31)],
    [(1.70, 27.31)],
    [(0.0, 15.57)],
    [(0.0, 5.05)],
    [(0.0, 16.39)],
    [(0.0, 16.53)],
    [(0.0, 13.55), (14.61, 25.52)],
    [(0.0, 4.08), (4.58, 12.34), (17.38, 27.31)],
    [(0.0, 27.31)],
    [(0.0, 18.58), (19.39, 27.27)],
    [(0.0, 9.29), (10.04, 27.31)],
    [(8.17, 14.08)],
    [(0.0, 20.69)],
    [(0.0, 21.38), (21.68, 24.10)],
    [(0.0, 14.91), (15.69, 20.96)],
    [(0.0, 16.28)],
    [(0.0, 27.31)],
    [(0.0, 10.89), (12.88, 16.30)],
    [(0.0, 20.22)],
    [(0.0, 17.14), (17.96, 27.31)],
    [(0.0, 11.65)],
    [(0.0, 18.38), (19.46, 21.55)],
    [(0.0, 23.69), (23.99, 27.31)],
    [(0.0, 14.32)],
    [(0.0, 14.12)],
    [(0.0, 19.20), (20.06, 24.38)],
    [(0.0, 6.80), (7.37, 16.83)],
    [(3.39, 15.45)],
    [(0.0, 24.18)],
    [(0.0, 19.60), (21.26, 24.86)],
    [(0.0, 27.31)],
    [(0.0, 16.58)],
    [(0.0, 27.31)],
    [(0.0, 13.03)],
    [(0.0, 27.31)],
    [(0.0, 27.31)],
    [(0.0, 3.19), (3.74, 19.24), (19.86, 27.31)],
    [(0.0, 4.46)],
    [(0.0, 20.58)],
    [(0.0, 27.31)],
    [(0.0, 25.34), (25.90, 27.31)],
    [(0.0, 27.31)],
    [(0.0, 27.31)],
    [(1.31, 11.59), (12.08, 15.75), (16.15, 27.31)],
    [(11.24, 27.31)],
    [(0.0, 27.31)],
    [(0.0, 2.41), (3.19, 16.21), (17.28, 27.31)],
    [(0.0, 26.09)],
    [(0.0, 5.41), (6.40, 23.46)],
    [(0.0, 2.14)],
    [(2.17, 14.77), (15.25, 16.36)],
    [(1.22, 9.63), (10.46, 22.33), (22.85, 24.70)],
    [(0.0, 26.97)],
    [(1.04, 5.17), (5.89, 17.89)],
    [(21.36, 27.31)],
    [(22.41, 27.31)],
    [(0.0, 20.443), (21.05, 23.34)],
    [(0.0, 18.77)],
    [(0.0, 8.82)],
    [(0.0, 7.98)],
    [(0.0, 27.31)],
    [(0.0, 1.61), (2.54, 11.26), (11.72, 15.80)],
    [(0.0, 27.31)],
    [(0.0, 27.31)],
    [(0.0, 26.08)],
    [(0.0, 23.54), (24.01, 27.31)],
    [(0.0, 15.66)],
    [(0.0, 12.58)],
    [(0.0, 26.61)],
    [(0.0, 3.85)],
    [(0.0, 25.32), (25.77, 27.31)],
    [(0.0, 14.05)],
    [(0.0, 27.31)],
    [(0.0, 16.51)],
    [(0.0, 13.59), (14.23, 18.38)],
    [(0.53, 3.59)],
    [(17.89, 27.31)],
    [(20.25, 27.31)],
    [(0.0, 23.71)],
    [(0.0, 27.31)],
    [(0.0, 7.64)],
    [(0.0, 22.89)],
    [(0.0, 8.71), (9.08, 11.38), (12.51, 14.89)],
    [(0.0, 11.15), (11.68, 14.77), (15.77, 18.26)],
    [(0.0, 27.31)],
    [(0.0, 21.97), (22.55, 27.31)],
    [(0.0, 14.65), (15.42, 27.31)],
    [(0.0, 12.70), (13.55, 16.88)],
    [(0.0, 27.31)],
    [(7.03, 26.57)],
    [(14.81, 27.31)],
    [(21.10, 27.31)],
    [(0.90, 3.10)],
    [(0.0, 16.46)],
    [(0.0, 4.90)],
    [(0.0, 26.65)],
    [(0.0, 27.31)],
    [(0.0, 13.91), (15.00, 21.63)],
    [(0.0, 11.84), (12.33, 16.95), (17.39, 18.89)],
    [(0.0, 14.71), (16.40, 26.37)],
    [(0.0, 20.76)],
    [(0.0, 16.49), (18.42, 27.31)],
    [(0.0, 7.19), (8.40, 16.61)],
    [(0.0, 20.96)],
    [(0.0, 2.31)],
    [(0.0, 14.80), (15.40,20.06)],
    [(0.0, 25.57)],
    [(0.0, 11.54), (12.43, 22.36), (22.80, 25.71)]]; 


# annotate
Lobs_ = []
tobs_ = []
for idx in eachindex(select_filename)
    tab = readdlm(data_dir * select_filename[idx]);
    varnames = convert(Vector{String}, tab[1,:]);
    tobs = tab[2:end,findfirst(varnames .== "time")] * 0.01;
    Lobs = tab[2:end,findfirst(varnames .== "length")] * DL;
    Lobs_idx, tobs_idx = filter_exp_traj(Lobs, tobs; select_time = select_time_all[idx]);
    Lobs_ = vcat(Lobs_, Lobs_idx)
    tobs_ = vcat(tobs_, tobs_idx)
end
Lobs_0129 = convert.(Vector{Float64}, Lobs_);
tobs_0129 = convert.(Vector{Float64}, tobs_);
# make 0129 and 0131 the same unit
tobs_0129 = tobs_0129 ./ 0.01 * DT;

In [7]:
Lobs_ = vcat(Lobs_0129, Lobs_0131);
tobs_ = vcat(tobs_0129, tobs_0131);

In [8]:
tobs_ = tobs_[ maximum.(Lobs_) .< 10.5 ];
Lobs_ = Lobs_[ maximum.(Lobs_) .< 10.5 ];

In [9]:
# Annotate division events and mother size (cell size at last division)
# and grandmother size (cell size at the division before last division)
isdiv_, dtobs_, Llast_, Llast2_ = annotate_exp_traj3(Lobs_, tobs_);

### trajectory

In [16]:
p_traj_obs = plot(; size=(750,300), xlim=(0,10), ylim=(0,13), xtick=collect(0:5:40), ytick=collect(0:5:15), bottommargin=-3Plots.mm,  leftmargin=-5Plots.mm, background_color=:transparent) # xlabel="Time", ylabel="Cell length"
for i in eachindex(Lobs_)
    plot!(p_traj_obs, tobs_[i], Lobs_[i], lc=:snow2, lw=0.2)
    # scatter!(p_traj_obs, tobs_[i][isdiv_[i]], Lobs_[i][isdiv_[i]], markercolor=:gray)
end
pj = 20
plot!(p_traj_obs, tobs_[pj], Lobs_[pj], lw = 0.4)
scatter!(p_traj_obs, tobs_[pj][isdiv_[pj]], Lobs_[pj][isdiv_[pj]], markercolor=:gray, ms=1, markerstrokewidth=0.2)
p_traj_obs
savefig("../figure/fig2/ecoli_trajectory.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_trajectory.pdf"

### correlation

In [22]:
using StatsBase
hedges = collect(3:0.25:10.5);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
h_obs = fit(Histogram, (vcat( [Lobs_[i][isdiv_[i]][1:end-1] for i in eachindex(Lobs_) ]...), vcat( [Lobs_[i][isdiv_[i]][2:end] for i in eachindex(Lobs_) ]...) ), (hedges,hedges));

h_obs_normalize = h_obs.weights / sum(h_obs.weights) / dhedges^2

svd_h_obs = svd(h_obs_normalize);

ll=@layout [a b]
p_h_obs = plot(; aspect_ratio=1, framestyle=:box, xlim=(3,9), ylim=(3,9), xtick=collect(3:3:10), ytick=collect(3:3:10))
heatmap!( p_h_obs, hbins, hbins, h_obs_normalize, clim=(0,0.4),c = :Blues_9,
colorbar_ticks=(),colorbar=false) 

plot(p_h_obs, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=5Plots.mm, background_color=:transparent)
savefig("../figure/fig2/ecoli_correlation.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_correlation.png"

In [24]:
p_S_obs = plot(; xlim=[0.5,10.5], ylim=[-0.1,1.05], ytick=[-0,0.5,1], xtick=collect(1:4:13), 
size=(300,300), bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
scatter!( p_S_obs, svd_h_obs.S ./ svd_h_obs.S[1], markercolor=:gray, ms=3, markershape=:hline,msw=4)
savefig("../figure/fig2/ecoli_svd.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_svd.pdf"

In [25]:
# mutual information
P_xy = h_obs.weights/sum(h_obs.weights)
P_y = sum(P_xy, dims=1)
P_x = sum(P_xy, dims=2)
is_nan = .!(isnan.(P_xy .* log.(P_xy./(P_x * P_y))))
mi = P_xy .* log.(P_xy./(P_x * P_y))
sum(mi[is_nan])


0.3677544327420437

### input data distribution

In [26]:
using StatsBase
hedges = collect(4:0.25:9);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
hh_obs = fit(Histogram, (vcat( [Lobs_[i] for i in eachindex(Lobs_) ]...), vcat( [Llast_[i] for i in eachindex(Llast_) ]...) ), (hedges,hedges));

hh_obs_normalize = hh_obs.weights / sum(hh_obs.weights) / dhedges^2

svd_hh_obs = svd(hh_obs_normalize);

ll=@layout [a b]
p_hh_obs = plot(; aspect_ratio=1, framestyle=:box, xlim=(4,9), ylim=(4,9), xtick=collect(5:2:10), ytick=collect(5:2:10))#, title="P(i,i-1)" 
heatmap!( p_hh_obs, hbins, hbins, hh_obs_normalize, clim=(0,0.2),c = :Blues_9,
colorbar_ticks=(),colorbar=false) 

plot(p_hh_obs, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=5Plots.mm, background_color=:transparent)
savefig("../figure/fig2/ecoli_datadist.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_datadist.png"

### inference

In [27]:
bf1 = [sChebyshevT{Float64}(),sChebyshevT{Float64}()]
n_func1 = [4, 4]
l01 = [5.5, 5.5]
Θ1, Σ1 = mgs_gen_bases( [vcat(Lobs_...), vcat(Llast_...)], n_func1, bf1, l01);
Θdiv1 = Θ1[vcat(isdiv_...),:];
Θdiv1 = sum(Θdiv1, dims=1)[:];

history1 = sparse_optimization_sweep(sparse_optimization_fg, Θ1, Θdiv1, vcat(dtobs_...); init_cutoff=0.0, coeffs0=1e-4, EPSILON=1E-7, MAX_ITERS=20);

===== start of epoch 1 =====
itr=1: -15055.467726677052
itr=2: -15079.698206122855
itr=3: -15078.20414984672
itr=4: -15077.432591800829
itr=5: -15076.934839980171
itr=6: -15076.897315962975
itr=7: -15076.387920282514
itr=8: -15076.387940940704
itr=1: -15055.467726677052
itr=2: -15079.698206122855
itr=3: -15078.20414984672
itr=4: -15077.432591800829
itr=5: -15076.934839980171
itr=6: -15076.897315962975
itr=7: -15076.387920282514
itr=8: -15076.387940940704
===== end of epoch 1 =====
error = -3.7003260722934384, nterms = 25, cutoff = 0.0
===== start of epoch 2 =====
itr=1: -15055.467726677052
itr=2: -15079.698206122855
itr=3: -15078.20414984672
itr=4: -15077.432591800829
itr=5: -15076.934839980171
itr=6: -15076.897315962975
itr=7: -15076.387920282514
itr=8: -15076.387940940704
itr=1: -15054.584039218029
itr=2: -15078.024829560141
itr=3: -15077.765253366453
itr=4: -15077.64912509944
itr=5: -15077.332910664334
itr=6: -15077.330846688463
itr=7: -15077.330743619816
itr=1: -15054.591117614087


In [28]:
save_inference_results([vcat(tobs_...) vcat(Lobs_...) vcat(Llast_...)], vcat(isdiv_...), vcat(dtobs_...), 
    [ [@sprintf "%s" typeof(bf1[i]) for i in 1:length(bf1)] n_func1 l01 ], history1; 
    data_type = "exp_ecoli", learn_type = "learn_memory1", output_dir = "./output/", 
    use_uid=true, save_traj=true
)

### Read in inference result

In [33]:
filepath1 = "./output/exp_ecoli_learn_memory1/24-04-11-T1617/"

traj = readdlm(filepath1 * "traj");
tobs = traj[:,1];
Ls1 = [traj[:,j] for j in 2:3];

dtobs = readdlm(filepath1 * "dts")[:];
isdiv = convert( BitVector, readdlm(filepath1 * "isdiv")[:] );

In [34]:
# rebuild inference basis function library
function rebuild_library_params(filepath)
    library_sum = readdlm(filepath * "library");
    bf_string = library_sum[:, 1];
    @show bf_string
    n_func = convert(Vector{Int}, library_sum[:,2]);
    l0 = convert(Vector{Float64}, library_sum[:,3]);
    return n_func, l0
end

n_func1, l01 = rebuild_library_params(filepath1);
bf1 = [sChebyshevT{Float64}(),sChebyshevT{Float64}()]
_, Σ1 = mgs_gen_bases(Ls1, n_func1, bf1, l01);

bf_string = Any["sChebyshevT{Float64}", "sChebyshevT{Float64}"]


In [35]:
# read in inference result
function rebuild_history(filepath)

    history_coeffs = readdlm(filepath * "history_coeffs");
    cache_summary  = readdlm(filepath * "history_summary");
    keynames = convert(Vector{String}, cache_summary[1,:] );
    history_metric = cache_summary[2:end,:];
    history = Dict(zip(Symbol.(keynames), [history_metric[:,i] for i = 1:length(keynames)]));

    return history, history_coeffs

end

history1, history_coeffs1 = rebuild_history(filepath1);

In [37]:
# find the maximum BIC score
_, imax1 = findmax( -history1[:error] .- 0.5*history1[:logdetH]/sum(dtobs) )

BIC = -history1[:error] .- 0.5*history1[:logdetH]/sum(dtobs)
plot(history1[:nterms], BIC, lw=2, lc=:dodgerblue4,
    xtick=[0,10,20],ytick=[0,2,4],ylim=(-1.1,4.0), size=(330,300), bottommargin=-6Plots.mm, leftmargin=-3Plots.mm,
    background_color=:transparent)
scatter!([history1[:nterms][imax1]], [BIC[imax1]], ms=2.5, mc=:dodgerblue4, shape=:rect)
savefig("../figure/fig2/ecoli_BIC.pdf")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_BIC.pdf"

In [38]:
# calculate the inferred λ
ll = collect(3.0:0.1:9.0);
ll_ = collect(3.0:0.1:9.0);
logλ_learn1 = mgs_rec_bases([ll, ll_], Σ1, history_coeffs1[:, imax1], bf1, l01);

pl1 = plot(;size=(300,300), bottommargin=-6Plots.mm, leftmargin=-3Plots.mm, background_color=:transparent);
heatmap!(pl1, ll, ll_, exp.(logλ_learn1'), clim=(0,13), 
xlim=(4,9), ylim=(4,9), aspect_ratio=1, framestyle=:box, 
c=:lapaz, xtick=[5,7,9],ytick=[5,7,9])
plot!(pl1, [4, 9], [4,9], lc=:black, ls=:dash, lw=1.0)
savefig("../figure/fig2/ecoli_memory1.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_memory1.png"

In [39]:

heatmap(reshape(history_coeffs1[:, imax1],5,5), c =:broc, ticks=false, axis=true, aspect_ratio =1, clim=(-3,3),
xlim=(0.5,5.5), ylim=(0.5,5.5),framestyle=:box,size=(300,300), background_color=:transparent, yflip = true
)
savefig("../figure/fig2/ecoli_coeffs_matrix.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_coeffs_matrix.png"

### Growth function g inference

In [53]:
# interpolation to smooth the growth function g
function interpolation(t, L, deg)
    A = ones(length(L), deg);
    for i = 2:deg
        A[:,i] = t.^(i-1);
    end
    coeff = A\L;
    D = zeros(length(L), deg);
    for i = 2:deg
        D[:,i] = (i-1) * t.^(i-2);
    end
    return coeff, D * coeff, A * coeff
end

interpolation (generic function with 1 method)

In [47]:
bf_g = Polynomial()
Lt_g = []
Diff_g = []
for Li in eachindex(Lobs_)
    pos = findall(isdiv_[Li]);
    for pi = 1:length(pos)-1
        _,diff, Lt = interpolation(tobs_[Li][pos[pi]+1:pos[pi+1]], Lobs_[Li][pos[pi]+1:pos[pi+1]], 5);
        push!(Lt_g, Lt);
        push!(Diff_g, diff);
    end
end

# linear regression to learn growth rate function g
Lt_g = convert(Vector{Vector{Float64}}, Lt_g);
Diff_g = convert(Vector{Vector{Float64}}, Diff_g);
Lt_g = vcat( [ Lt_g[i]  for i in eachindex(Lt_g)]... );
Diff_g = vcat( [Diff_g[i] for i in eachindex(Diff_g) ]... );
Theta_g = gen_bases( Lt_g,  2);
coeffs_g = Theta_g[:,[1,2]] \ Diff_g

2-element Vector{Float64}:
 -0.3582207294178681
  1.95504395797935

In [52]:
histogram2d(Lt_g, Diff_g, framestyle=:box, normalize=:pdf, show_empty=true, size=(300,300), background_color=:transparent, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, 
xlim = (2,8), ylim = (-10,30), xtick =[2, 5, 8], ytick=[0, 15, 30], bins=(2:0.4:8, -10:2:30), c=cgrad(:tempo, rev=false), clim=(0,0.1)) # bins=(2:0.6:8, 0:3:30),aspect_ratio=0.2, c=:lapaz,
plot!([2,8], coeffs_g[1] .+ coeffs_g[2]*[2,8], lw=1.0, lc=:black, ls=:dash, colorbar=:false)
savefig("../figure/fig2/ecoli_infered_g.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_infered_g.png"

### Cut size function h inference

In [55]:
# linear regression to learn cut size function h
bf_h = Polynomial()
Y_h = vcat( [Lobs_[i][isdiv_[i]] .- Lobs_[i][findall(isdiv_[i]) .+  1]  for i in eachindex(Lobs_)]... );
Theta_h = gen_bases( vcat( [Lobs_[i][isdiv_[i]] for i in eachindex(Lobs_) ]... ), 1);

coeffs_h = Theta_h\ Y_h
coeffs_h[abs.(coeffs_h) .< 0.02] .= 0.0

0-element view(::Vector{Float64}, Int64[]) with eltype Float64

In [57]:
histogram2d( vcat( [Lobs_[i][isdiv_[i]] for i in eachindex(Lobs_) ]... ), Y_h, framestyle=:box, normalize=:pdf, show_empty=true, size=(300,300), background_color=:transparent, bottommargin=-6Plots.mm, leftmargin=-2Plots.mm,
xlim = (3,9), ylim = (0,6), xtick =[3, 6, 9], ytick=[0, 3, 6], bins=(3:0.2:9, 0:0.2:6), c=cgrad(:tempo, rev=false), clim=(0,1.2)) # bins=(2:0.6:8, 0:3:30),aspect_ratio=0.2, c=:lapaz,)
plot!([0, 20], coeffs_h[1] .+ coeffs_h[2] * [0, 20].^1, lw=1.0, lc=:black, ls=:dash)
savefig("../figure/fig2/ecoli_infered_h.png")



"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_infered_h.png"

### Resimulation

In [58]:
function f(du, u, p, t)
    bf_g = p[1]
    coeffs_g = p[2]
    du[1] = 0.0
    for n = 1:length(coeffs_g)
        du[1] += coeffs_g[n]*bf_g(n-1, u[1])
    end
end

function rate_memory(u, p, t)
    Σ1 = p[5]
    coeffs = p[6]
    bf1 = p[7]
    l01 = p[8]
    ln_rate = mgs_rec_bases([[u[1]], [u[2]]], Σ1, coeffs, bf1, l01);
    return exp(ln_rate[1])
end

function affect!(integrator)
    integrator.u[2] = integrator.u[1]
    bf_h = integrator.p[3]
    coeffs_h = integrator.p[4]
    u_old = integrator.u[1]
    for n = 1:length(coeffs_g)
        integrator.u[1] -= coeffs_h[n] * bf_h(n-1, u_old)
    end
end

affect! (generic function with 1 method)

In [64]:
# resimulate data and save or read in your resimulated data
# resimulate data based on your inferred parameters
Lresult = []
tresult = []
for i in eachindex(Lobs_)
    vrj = VariableRateJump(rate_memory, affect!)
    oprob = ODEProblem(f, [Lobs_[i][1];2 * Lobs_[i][1]], (0.0,maximum(tobs_[i])), (bf_g, coeffs_g, bf_h, coeffs_h, Σ1, history_coeffs1[:, imax1], bf1, l01))
    jprob = JumpProblem(oprob, Direct(), vrj)
    sol = solve(jprob,Tsit5(),dtmax=0.01);
    L_r = Array(sol')[:,1];
    t_r = sol.t;
    push!(Lresult, L_r);
    push!(tresult, t_r);
end

Lresult = convert(Vector{Vector{Float64}}, Lresult);
tresult = convert(Vector{Vector{Float64}}, tresult);

isdiv_result = []
for i in eachindex(Lresult)
    isdiv_r, _ = annotate_sim_traj(Lresult[i], tresult[i])
    push!(isdiv_result, isdiv_r);
end

writedlm("../figure/fig2/ecoli_resim_data.txt", [Lresult, tresult], '\t')

In [68]:
# read in resimulated data
using JSON
resim_data = readdlm("../figure/fig2/ecoli_resim_data.txt", '\t')

Lresult = [Float64.(JSON.parse(resim_data[1, i])) for i in 1:265];
tresult = [Float64.(JSON.parse(resim_data[2, i])) for i in 1:265];

isdiv_result = []
for i in eachindex(Lresult)
    isdiv_r, _ = annotate_sim_traj(Lresult[i], tresult[i])
    push!(isdiv_result, isdiv_r);
end

In [70]:
p_traj_obs = plot(; size=(750,300), xlim=(0,10), ylim=(0,13), xtick=collect(0:5:40), ytick=collect(0:5:15), bottommargin=-3Plots.mm,  leftmargin=-5Plots.mm, background_color=:transparent) # xlabel="Time", ylabel="Cell length"
for i =1:50
    plot!(p_traj_obs, tresult[i], Lresult[i], lc=:snow2, lw=0.2)
    # scatter!(p_traj_obs, tobs_[i][isdiv_[i]], Lobs_[i][isdiv_[i]], markercolor=:gray)
end
pj=11
plot!(p_traj_obs, tresult[pj], Lresult[pj], lw = 0.4, lc=:seagreen)
scatter!(p_traj_obs, tresult[pj][isdiv_result[pj]], Lresult[pj][isdiv_result[pj]], markercolor=:seagreen, ms=1, markerstrokewidth=0.2)
p_traj_obs
savefig("../figure/fig2/ecoli_trajectory_resim.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_trajectory_resim.pdf"

In [72]:
using StatsBase
hedges = collect(3:0.25:10.5);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
h_result = fit(Histogram, (vcat( [Lresult[i][isdiv_result[i]][1:end-1] for i in eachindex(Lresult) ]...), vcat( [Lresult[i][isdiv_result[i]][2:end] for i in eachindex(Lresult) ]...) ), (hedges,hedges));

h_result_normalize = h_result.weights / sum(h_result.weights) / dhedges^2

svd_h_result = svd(h_result_normalize);

p_h_result = plot(; aspect_ratio=1, framestyle=:box, xlim=(3,9), ylim=(3,9), xtick=collect(3:3:10), ytick=collect(3:3:10))#, title="P(i,i-1)" 
heatmap!( p_h_result, hbins, hbins, h_result_normalize, clim=(0,0.4),c = :Blues_9,
colorbar_ticks=()) 

plot(p_h_result, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
savefig("../figure/fig2/ecoli_correlation_resim.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_correlation_resim.png"

In [82]:
p_S_result = plot(; xlim=[0.5,10.5], ylim=[-0.1,1.05], ytick=[0,0.5,1], xtick=collect(1:4:13), 
size=(300,300), bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
scatter!( p_S_result, svd_h_obs.S ./ svd_h_obs.S[1], markercolor=:gray, ms=4, markershape=:hline,msw=3)
scatter!( p_S_result, svd_h_result.S ./ svd_h_result.S[1], markercolor=:seagreen, ms=3.5, markershape=:circle,msw=1, msc=:seagreen)

savefig("../figure/fig2/ecoli_svd_resim.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_svd_resim.pdf"

In [87]:
# mutual information
P_xy = h_result.weights/sum(h_result.weights)
P_y = sum(P_xy, dims=1)
P_x = sum(P_xy, dims=2)
is_nan = .!(isnan.(P_xy .* log.(P_xy./(P_x * P_y))))
mi = P_xy .* log.(P_xy./(P_x * P_y))
sum(mi[is_nan])

0.3925435735014654

###  histogram

In [85]:
p1 = plot(;xlim=(3,9),  xtick=collect(3:3:10),ylim=(-0.02,0.6),ytick=[0,0.3,0.6])
stephist!( p1, vcat( [Lobs_[i][isdiv_[i]] for i in eachindex(Lobs_) ]...), bins=collect(3:0.25:9), normalize=:pdf, lw=0.2#, xlabel="Cell length", ylabel="PDF"
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)

stephist!( p1, vcat( [Lresult[i][isdiv_result[i]] for i in eachindex(Lresult) ]...), normalize=:pdf, bins=collect(3:0.25:9), lc=:seagreen, lw=0.2)
savefig("../figure/fig2/ecoli_histogram_divsize.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_histogram_divsize.pdf"

In [86]:
p2 = plot()
stephist!( p2, vcat( diff.([tobs_[i][isdiv_[i]] for i in eachindex(tobs_)]) ... ), normalize=:pdf,
        xlim=(0,0.82), xtick=collect(0:0.2:0.8),ylim=(-0.02,8),ytick=[0,4,8], bins=collect(0:0.05:0.8), lw=0.2
        , bottommargin=-6Plots.mm, leftmargin=0Plots.mm, background_color=:transparent)
        #,ylabel="PDF", xlabel="Generation time" )
stephist!( p2, vcat( diff.([tresult[i][isdiv_result[i]] for i in eachindex(tresult)]) ... ), normalize=:pdf
, lc=:seagreen, bins=collect(0:0.05:0.8), lw=0.2)

savefig("../figure/fig2/ecoli_histogram_gentime.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/fig2/ecoli_histogram_gentime.pdf"